## 参数管理
主要介绍如下内容
- 访问参数，用于调试、诊断和可视化
- 参数初始化
- 在不同模型组件间共享参数
我们首先看一下具有单隐藏层的多层感知机

In [1]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,1))
X = torch.rand(size=(2,4))
net(X)

tensor([[ 0.1210],
        [-0.0386]], grad_fn=<AddmmBackward0>)

### 参数访问
我们从已有模型中访问参数。 当通过Sequential类定义模型时， 我们可以通过索引来访问模型的任意层。 这就像模型是一个列表一样，每层的参数都在其属性中。 如下所示，我们可以检查第二个全连接层的参数。

In [2]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[ 0.1877,  0.1334, -0.3105,  0.1656,  0.3242, -0.0402, -0.2738,  0.0192]])), ('bias', tensor([0.3320]))])


输出的结果告诉我们一些重要的事情： 首先，这个全连接层包含两个参数，分别是该层的权重和偏置。 两者都存储为单精度浮点数（float32）。 注意，参数名称允许唯一标识每个参数，即使在包含数百个层的网络中也是如此。
##### 目标参数
注意，每个参数都表示为参数类的一个实例。 要对参数执行任何操作，首先我们需要访问底层的数值。 有几种方法可以做到这一点。有些比较简单，而另一些则比较通用。 下面的代码从第二个全连接层（即第三个神经网络层）提取偏置， 提取后返回的是一个参数类实例，并进一步访问该参数的值。

In [4]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)
# 在上面这个网络中，由于我们还没有调用反向传播，所以参数的梯度处于初始状态
net[2].weight.grad == None  

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.3320], requires_grad=True)
tensor([0.3320])


True

#### 一次性访问所有参数
当我们需要对所有参数执行操作时，逐个访问它们可能会很麻烦。 当我们处理更复杂的块（例如，嵌套块）时，情况可能会变得特别复杂， 因为我们需要递归整个树来提取每个子块的参数。 下面，我们将通过演示来比较访问第一个全连接层的参数和访问所有层。

In [7]:
print(*[(name,param.shape) for name,param in net[0].named_parameters()])
print(*[(name,param.shape) for name,param in net.named_parameters()])
# 另一种访问网络参数的方式
net.state_dict()['2.bias'].data

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


tensor([0.3320])

#### 从嵌套块收集参数
 我们首先定义一个生成块的函数（可以说是“块工厂”），然后将这些块组合到更大的块中。

In [8]:
def block1():
    return nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,4),nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net.add_module(f'block{i}',block1())
    return net

rgnet = nn.Sequential(block2(),nn.Linear(4,1))  
rgnet(X)

tensor([[0.4994],
        [0.4994]], grad_fn=<AddmmBackward0>)

In [9]:
# 我们查看如何工作
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [10]:
# 因为层是分层嵌套的，我们也可以像嵌套列表索引一样访问他们，下面我们访问第一个主要的块中、第二个子块的第一层的偏置项
rgnet[0][1][0].bias.data

tensor([-0.1628,  0.2747,  0.2560,  0.0712, -0.4918, -0.2725,  0.0075,  0.0403])

### 参数初始化
默认情况下，PyTorch会根据一个范围均匀地初始化权重和偏置矩阵， 这个范围是根据输入和输出维度计算出的。 PyTorch的nn.init模块提供了多种预置初始化方法。
#### 内置初始化
让我们首先调用内置的初始化器。 下面的代码将所有权重参数初始化为标准差为0.01的高斯随机变量， 且将偏置参数设置为0。

In [11]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight,mean=0,std=0.01) #就地修改
        nn.init.zeros_(m.bias) # 权重初始化，将偏置初始化并将其他元素都设置为0
net.apply(init_normal)
net[0].weight.data[0],net[0].bias.data[0]

(tensor([-0.0036,  0.0036, -0.0039, -0.0100]), tensor(0.))

In [12]:
# 我们还可以即将所有参数初始化为给定的常数，例如6
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight,6)
        nn.init.zeros_(m.bias)  
net.apply(init_constant)
net[0].weight.data[0],net[0].bias.data[0]

(tensor([6., 6., 6., 6.]), tensor(0.))

In [15]:
# 我们还可以对某些块应用不同的初始化方法。 例如，下面我们使用Xavier初始化方法初始化第一个神经网络层， 
# 然后将第三个神经网络层初始化为常量值42。
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight,42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data[0])

tensor([0.6463, 0.1357, 0.6727, 0.0338])
tensor([42., 42., 42., 42., 42., 42., 42., 42.])


#### 自定义初始化
我们使用以下的分布为任意权重参数w定义初始化方法:$$w\sim\begin{cases}U(5,10)&\text{可能性}\frac14\\0&\text{可能性}\frac12\\U(-10,-5)&\text{可能性}\frac14\end{cases}$$ 同样我们实现一个my_init函数来应用到net

In [18]:
def my_init(m):
    if type(m) == nn.Linear:
        print("init",*[(name,param.shape) for name,param in m.named_parameters()][0])
        nn.init.uniform_(m.weight,-10,10) # 初始化权重 -10 到 10 之间
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

init weight torch.Size([8, 4])
init weight torch.Size([1, 8])


tensor([[ 8.8810,  0.0000,  5.7452, -0.0000],
        [-0.0000,  0.0000, -7.7394,  0.0000]], grad_fn=<SliceBackward0>)

In [19]:
# 注意我们可以直接设置参数
net[0].weight.data[:] += 1
net[0].weight.data[0,0] =42
net[0].weight.data[0]

tensor([42.0000,  1.0000,  6.7452,  1.0000])

### 参数绑定
有时我们希望在多个层间共享参数： 我们可以定义一个稠密层，然后使用它的参数来设置另一个层的参数。


In [20]:
# 我们需要给共享层一个名称，以便可以引用它的参数
shared = nn.Linear(8,8)
net = nn.Sequential(nn.Linear(4,8),nn.ReLU(),shared,nn.ReLU(),shared,nn.ReLU(),nn.Linear(8,1))      
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0,0] == 100
# 确保他们实际上是同一个对象，而不是只有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


这个例子表明第三个和第五个神经网络层的参数是绑定的。 它们不仅值相等，而且由相同的张量表示。 因此，如果我们改变其中一个参数，另一个参数也会改变。 这里有一个问题：当参数绑定时，梯度会发生什么情况？ 答案是由于模型参数包含梯度，因此在反向传播期间第二个隐藏层 （即第三个神经网络层）和第三个隐藏层（即第五个神经网络层）的梯度会加在一起。